# ml4seeding — Package Orientation

`ml4seeding` is a Python package for **microsite classification from high-resolution drone imagery**, supporting precision aerial reforestation (aerial seeding) in British Columbia.

This notebook is a guided tour of the package's capabilities. It runs **entirely on synthetic data** — no drone imagery and no TensorFlow installation required.

## Package overview

| Module | Purpose | Needs TF? |
|--------|---------|-----------|
| `orthomosaic` | Analyze RGB orthomosaics: sharpness, edge density, vegetation indices, summary stats | No |
| `pseudo_ortho` | Build pseudo-orthomosaics from DJI drone images via GPS stitching | No |
| `tiling` | Tile GeoTIFFs into fixed-size PNG tiles with georeferenced metadata | No |
| `grid` | Camouflage grid overlay for annotation quality control | No |
| `preprocessing` | Image–mask matching, spatial train/val/test splitting, tile filtering and oversampling | No |
| `metrics` | Segmentation evaluation: per-class IoU, mean IoU, foreground IoU, accuracy | No |
| `models` | U-Net architecture definition and builder | Yes (builder only) |
| `losses` | Loss functions for training | Yes |
| `training` | Training pipeline | Yes |

Sections 1–8 below exercise the TF-free modules with synthetic arrays.

## 1. Package import and version check

The package exposes its version via `ml4seeding.__version__`. Only `numpy` and `matplotlib` are needed for this notebook.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ml4seeding import __version__

print('ml4seeding version:', __version__)

## 2. Orthomosaic analysis demo

The `orthomosaic` module provides pure-array helpers for assessing orthomosaic quality and content:

- **`laplacian_variance(gray)`** — sharpness measure (higher = sharper; <500 very blurry, >4000 sharp)
- **`edge_density(gray)`** — fraction of pixels whose gradient magnitude is in the top 5 %
- **`rgb_indices(rgb)`** — vegetation indices ExG, VARI, GRVI from an `(H, W, 3)` RGB array
- **`summ_stats(x)`** — robust summary statistics (min, p2, p50, p98, max, mean, std)

We build a synthetic "orthomosaic": reddish-brown bare-soil background with green vegetation discs, plus texture noise. Vegetation pixels have **G > R**, so they should light up in the greenness indices.

In [ ]:
rng = np.random.default_rng(42)
H, W = 400, 600

# Base layer: bare soil (reddish-brown, R > G > B)
r = np.full((H, W), 150.0, dtype=np.float32)
g = np.full((H, W), 110.0, dtype=np.float32)
b = np.full((H, W), 80.0, dtype=np.float32)

# Vegetation discs (green, G > R)
yy, xx = np.mgrid[0:H, 0:W]
for cy, cx, rad in [(120, 150, 70), (300, 380, 90), (180, 480, 55), (80, 420, 45)]:
    m = (yy - cy) ** 2 + (xx - cx) ** 2 <= rad ** 2
    r[m], g[m], b[m] = 60.0, 140.0, 50.0

# Add texture noise so sharpness metrics have something to measure
r += rng.normal(0, 12, (H, W))
g += rng.normal(0, 12, (H, W))
b += rng.normal(0, 10, (H, W))

rgb = np.clip(np.stack([r, g, b], axis=-1), 0, 255).astype(np.uint8)
print('Synthetic orthomosaic:', rgb.shape, rgb.dtype)

In [ ]:
from ml4seeding.orthomosaic import (
    edge_density,
    laplacian_variance,
    rgb_indices,
    summ_stats,
)

# Grayscale via the same luminance weights used inside the package
gray = 0.2989 * rgb[..., 0] + 0.5870 * rgb[..., 1] + 0.1140 * rgb[..., 2]

sharp = laplacian_variance(gray)
edges = edge_density(gray)
print(f'Laplacian variance (sharpness): {sharp:.1f}')
print(f'Edge density:                   {edges:.4f}')

# Vegetation indices: ExG = 2G-R-B, VARI = (G-R)/(G+R-B), GRVI = (G-R)/(G+R)
exg, vari, grvi = rgb_indices(rgb)

for name, idx in [('ExG', exg), ('VARI', vari), ('GRVI', grvi)]:
    s = summ_stats(idx)
    print(f"{name:5s} mean={s['mean']:+.3f}  p50={s['p50']:+.3f}")


## 3. Vegetation indices visualization

The three indices returned by `rgb_indices` emphasize green vegetation in slightly different ways:

- **ExG** (Excess Green): `2G − R − B` — simple and robust for RGB imagery
- **VARI** (Visible Atmospherically Resistant Index): `(G − R) / (G + R − B)`
- **GRVI** (Green–Red Vegetation Index): `(G − R) / (G + R)`

All three should show positive values (green in the diverging colormap) over the vegetation discs.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(rgb)
axes[0].set_title('Synthetic RGB ortho')

for ax, (name, idx) in zip(axes[1:], [('ExG', exg), ('VARI', vari), ('GRVI', grvi)]):
    im = ax.imshow(idx, cmap='RdYlGn')
    ax.set_title(name)
    fig.colorbar(im, ax=ax, fraction=0.046)

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Tiling demo

The `tiling` module cuts a GeoTIFF into fixed-size PNG tiles with JSON sidecar metadata. Two pure helpers drive its invalid-pixel filtering:

- **`invalid_fraction(tile, black_thresh=5)`** — fraction of pixels that are near-black in *all* channels (nodata/black collar regions)
- **`compute_valid_bbox(img, black_thresh=5)`** — bounding box `(x0, y0, x1, y1)` of valid pixels

`tile_geotiff` keeps only full tiles whose invalid fraction is below `skip_invalid_fraction` (default 0.70). Here we simulate the logic: a synthetic scene with a black border, tiled on a grid with overlap — tiles are kept or skipped based on their invalid fraction.

In [ ]:
from ml4seeding.tiling import compute_valid_bbox, invalid_fraction

# Synthetic scene: valid imagery inside, near-black 'nodata collar' outside
ortho = np.zeros((400, 640, 3), dtype=np.uint8)
ortho[120:280, 200:440] = rgb[100:260, 200:440]

print(f'Invalid fraction of whole scene: {invalid_fraction(ortho):.3f}')

x0, y0, x1, y1 = compute_valid_bbox(ortho)
print(f'Valid bounding box: x0={x0}, y0={y0}, x1={x1}, y1={y1}')

# Simulate the tiling decision loop (as in tile_geotiff)
tile_size, overlap, skip_frac = 128, 32, 0.70
step = tile_size - overlap
kept, skipped = [], []
for top in range(0, ortho.shape[0] - tile_size + 1, step):
    for left in range(0, ortho.shape[1] - tile_size + 1, step):
        tile = ortho[top:top + tile_size, left:left + tile_size]
        inv = invalid_fraction(tile)
        (kept if inv < skip_frac else skipped).append((left, top, inv))

print(f'Kept {len(kept)} tiles, skipped {len(skipped)} tiles '
      f'(tile_size={tile_size}, overlap={overlap}, skip_invalid_fraction={skip_frac})')

In [ ]:
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(ortho)
for left, top, inv in kept:
    ax.add_patch(mpatches.Rectangle((left, top), tile_size, tile_size,
                                    fill=False, edgecolor='lime', lw=1.5))
for left, top, inv in skipped:
    ax.add_patch(mpatches.Rectangle((left, top), tile_size, tile_size,
                                    fill=False, edgecolor='red', lw=1, ls='--'))
ax.legend(handles=[
    mpatches.Patch(fill=False, edgecolor='lime', label=f'kept ({len(kept)})'),
    mpatches.Patch(fill=False, edgecolor='red', label=f'skipped ({len(skipped)})'),
], loc='upper right')
ax.set_title('Tiling: green = kept (invalid fraction < 0.70), red = skipped')
ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Grid overlay demo

The `grid` module supports **annotation quality control**: a camouflage grid is blended over each tile so annotators can judge distances on the ground.

- **`grid_spacing_px(pixel_size_m, grid_cm)`** — converts a ground spacing (cm) to pixels using the ground sample distance
- **`draw_grid_camo(img, spacing, ...)`** — draws grid lines colored by the *mean color of neighbouring pixels* and alpha-blends them, so the grid is visible without obscuring the imagery

Example: at 0.5 cm/px GSD, a 10 cm ground grid is `0.10 / 0.005 = 20` px.

In [ ]:
from ml4seeding.grid import draw_grid_camo, grid_spacing_px

spacing = grid_spacing_px(pixel_size_m=0.005, grid_cm=10.0)
print(f'Ground grid 10 cm at 0.005 m/px -> spacing = {spacing} px')

# A 128x128 tile cropped from the synthetic ortho
tile = rgb[100:228, 100:228].copy()
gridded = draw_grid_camo(tile, spacing=spacing, thickness=1, band=6, alpha=0.8)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(tile)
axes[0].set_title('Original tile')
axes[1].imshow(gridded)
axes[1].set_title(f'Camouflage grid (spacing={spacing}px)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Data preprocessing demo

The `preprocessing` module prepares tile datasets for training:

- **`spatial_split(images, masks)`** — sorts tiles by their X coordinate (parsed from filenames like `x10240_y1920_s512.png`) and splits into train/val/test. Spatially contiguous blocks reduce **spatial leakage**: adjacent tiles share overlap and look similar, so a random split would leak information between train and validation sets.
- **`analyze_mask(path)`** — unique classes and per-class pixel ratios of a segmentation mask
- **`filter_trivial_tiles(...)`** — drops low-information tiles dominated by the `poor` class (default: no `good`/`fair` pixels and ≥95 % `poor`)
- **`oversample_minority_tiles(...)`** — replicates tiles containing minority classes (`good`, `fair`) to balance the training set

First: a spatial split over synthetic tile filenames (no files needed).

In [ ]:
from ml4seeding.preprocessing import extract_x_coord, spatial_split

# Synthetic tile index: 3 rows x 10 columns of 512 px tiles
names = [f'x{x * 512}_y{y * 512}_s512' for y in range(3) for x in range(10)]
images = [f'images/{n}.png' for n in names]
masks = [f'masks/{n}_mask.png' for n in names]

print('X coordinate parsed from', names[7], '->', extract_x_coord(names[7]))

splits = spatial_split(images, masks, train_frac=0.70, val_frac=0.15)
for subset, (imgs, msks) in splits.items():
    xs = [extract_x_coord(p) for p in imgs]
    print(f'{subset:5s}: {len(imgs):2d} tiles, x-range {min(xs)}-{max(xs)}')

Next: mask analysis, trivial-tile filtering, and minority oversampling. These helpers read mask PNGs from disk, so we write a few tiny synthetic masks to a temporary directory (classes: 0 = background, 1 = good, 2 = fair, 3 = poor).

In [ ]:
import tempfile
from pathlib import Path

from PIL import Image

from ml4seeding.preprocessing import (
    analyze_mask,
    filter_trivial_tiles,
    oversample_minority_tiles,
)

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    specs = {
        'all_poor': np.full((32, 32), 3, dtype=np.uint8),          # 100% poor -> filtered out
        'mixed': np.tile(np.array([0, 1, 2, 3], dtype=np.uint8), (32, 8)),  # all classes
        'good_only': np.where(np.indices((32, 32)).sum(0) < 32, 1, 0).astype(np.uint8),
        'fair_only': np.where(np.indices((32, 32)).sum(0) < 32, 2, 0).astype(np.uint8),
    }
    img_list, msk_list = [], []
    for name, arr in specs.items():
        Image.fromarray(arr).save(tmp / f'{name}_mask.png')
        (tmp / f'{name}.png').touch()  # stand-in image file (never read)
        img_list.append(str(tmp / f'{name}.png'))
        msk_list.append(str(tmp / f'{name}_mask.png'))

    classes, ratios = analyze_mask(msk_list[1])
    print('analyze_mask(mixed): classes =', sorted(classes),
          ', ratios =', {k: round(float(v), 2) for k, v in ratios.items()})

    f_img, f_msk, removed = filter_trivial_tiles(img_list, msk_list)
    print(f'filter_trivial_tiles: {len(img_list)} -> {len(f_img)} tiles ({removed} removed)')

    o_img, o_msk, ex_good, ex_fair = oversample_minority_tiles(
        f_img, f_msk, oversample_good=3, oversample_fair=2)
    print(f'oversample_minority_tiles: {len(f_img)} -> {len(o_img)} tiles '
          f'(+{ex_good} good, +{ex_fair} fair)')

## 7. Evaluation metrics demo

The `metrics` module evaluates segmentation predictions against ground truth:

- **`iou_per_class`** — Intersection-over-Union for each class
- **`mean_iou`** — mean IoU over all classes
- **`foreground_mean_iou`** — mean IoU excluding background (class 0)
- **`accuracy`** — overall pixel accuracy
- **`evaluate_segmentation`** — all of the above in one dict, with named classes (default: background / good / fair / poor)

We synthesize a 4-class ground truth and a "prediction" made by corrupting 15 % of the pixels.

In [ ]:
from ml4seeding.metrics import (
    accuracy,
    evaluate_segmentation,
    foreground_mean_iou,
    iou_per_class,
    mean_iou,
)

rng = np.random.default_rng(0)
y_true = rng.choice(4, size=(256, 256), p=[0.5, 0.2, 0.15, 0.15])

y_pred = y_true.copy()
flip = rng.random(y_true.shape) < 0.15
y_pred[flip] = rng.choice(4, size=int(flip.sum()))

print('Per-class IoU:  ', {k: round(v, 3) for k, v in iou_per_class(y_true, y_pred).items()})
print(f'Mean IoU:        {mean_iou(y_true, y_pred):.3f}')
print(f'Foreground mIoU: {foreground_mean_iou(y_true, y_pred):.3f}')
print(f'Accuracy:        {accuracy(y_true, y_pred):.3f}')
print()
for k, v in evaluate_segmentation(y_true, y_pred).items():
    print(f'{k:22s} {v:.3f}')

## 8. U-Net architecture overview

The `models` module defines a U-Net for 4-class microsite segmentation. The module is **importable without TensorFlow** — the `ARCHITECTURE` constants document the reference design, and only the builder functions (`unet_model`, `conv_block`, `upsampling_block`) require TF (install the `ml` extra to get it).

Design highlights:

- **Encoder**: 4 blocks of Conv2D → BatchNorm → Conv2D → BatchNorm → Dropout → MaxPool, doubling filters each level (32 → 256)
- **Bottleneck**: 512 filters, dropout 0.3, no pooling
- **Decoder**: 4 upsampling blocks (Conv2DTranspose + skip-connection concatenation + convs)
- **Head**: final 3×3 conv then 1×1 conv with softmax over 4 classes
- Input: 512×512×3 tiles (matching the default `tile_geotiff` output)

In [ ]:
import json

from ml4seeding.models import ARCHITECTURE

print(json.dumps(ARCHITECTURE, indent=2))

# The builder needs TensorFlow; degrade gracefully when it is absent.
try:
    from ml4seeding.models import unet_model
    model = unet_model(
        input_size=tuple(ARCHITECTURE['input_shape']),
        n_filters=ARCHITECTURE['base_filters'],
        n_classes=ARCHITECTURE['num_classes'],
    )
    print(f'Built U-Net with {model.count_params():,} parameters')
except ImportError as e:
    print('TensorFlow not available:', e)
    print("Install the ML extra with: pip install 'ml4seeding[ml]'")

## 9. Next steps

This notebook used synthetic arrays throughout. The end-to-end pipeline on real drone data looks like this:

1. **Build a pseudo-orthomosaic** from raw DJI images (GPS stitching): `pseudo_ortho.get_lat_lon` → `latlon_to_local_xy_m` → `estimate_spacing_m` → `select_images_for_coverage` → `build_pseudo_ortho`
2. **Analyze the orthomosaic**: `orthomosaic.analyze_rgb_ortho(ortho_path, out_dir)` writes a JSON quality report (band stats, vegetation indices, sharpness, black/nodata fractions) plus a quicklook PNG
3. **Tile it**: `tiling.tile_geotiff(ortho_path, out_dir, tile_size=512, overlap=128)` → PNG tiles + georeferenced JSON sidecars + `tiles_index.json`
4. **Grid overlay for annotation QC**: `grid.add_grid_to_tiles(tiles_dir, meta_dir, out_dir, grid_cm=10)`
5. **Prepare the dataset**: `preprocessing.load_image_mask_lists` → `spatial_split` → `filter_trivial_tiles` → `oversample_minority_tiles`
6. **Train** (needs `pip install 'ml4seeding[ml]'`): `training` pipeline with `models.unet_model` and `losses`
7. **Evaluate**: `metrics.evaluate_segmentation(y_true, y_pred)`

### CLI

The same pipeline is exposed through the `ml4seeding` command-line interface:

```bash
ml4seeding --version
ml4seeding info
ml4seeding ortho analyze ORTHO_PATH --out-dir OUTPUT_DIR
ml4seeding pseudo-ortho build INPUT_DIR --out-dir OUTPUT_DIR
ml4seeding tile ORTHO_PATH --out-dir OUTPUT_DIR [--tile-size 512] [--overlap 128]
ml4seeding train CONFIG_PATH [--epochs 50] [--batch-size 4]
ml4seeding evaluate MODEL_PATH --image-dir IMAGES --mask-dir MASKS
```

See the project README and the `examples/` directory for notebooks that run this pipeline on real drone imagery.